# Notebook 49 — Adaptive Infrastructure

**Adaptive infrastructure turns resource allocation into deployable closed-loop serving.**

This notebook follows Notebook 43. The first systems arc produced an operating-regime controller. Notebook 49 turns that controller into deployable infrastructure: routing, telemetry, autoscaling, replica placement, saturation handling, and feedback.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, Image, FileLink

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 160,
    "font.size": 13,
    "axes.titlesize": 24,
    "axes.labelsize": 15,
    "legend.fontsize": 12,
})

In [ ]:
def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12, lw=1.7):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.035,rounding_size=0.045",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, lw=1.8, ms=13, connectionstyle="arc3,rad=0.0"):
    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=ms,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle
    ))

def flow_diagram(title, labels, filename, caption=None, figsize=(14, 5), y=0.52):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, pad=20)
    n = len(labels)
    w = min(0.15, 0.82 / n)
    h = 0.19
    xs = np.linspace(0.06, 0.94 - w, n)
    for i, (x, label) in enumerate(zip(xs, labels)):
        rounded_box(ax, (x, y), w, h, label, fontsize=12)
        if i < n - 1:
            arrow(ax, (x + w, y + h/2), (xs[i+1] - 0.01, y + h/2))
    if caption:
        rounded_box(ax, (0.25, 0.15), 0.5, 0.14, caption, fontsize=12)
    savefig(filename)

## 1. Repository transition

Notebook 49 starts the infrastructure arc. Notebook 43 selected resource budgets by operating regime; Notebook 49 maps those policies onto a serving fabric.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive infrastructure continues the second systems arc", pad=24)

top = [
    ("00\nContext", 0.04), ("07\nVerification\nResource", 0.18), ("13\nConfidence\nScheduling", 0.32),
    ("17\nSemi-AR\nDrafting", 0.46), ("23\nThroughput\nObjective", 0.60),
    ("29\nHardware\nConstraints", 0.74), ("37\nOperating\nRegimes", 0.88)
]
w, h = 0.11, 0.15
for i, (label, x) in enumerate(top):
    rounded_box(ax, (x, 0.64), w, h, label, fontsize=10)
    if i < len(top)-1:
        arrow(ax, (x+w, 0.715), (top[i+1][1]-0.01, 0.715), ms=10)

ax.plot([0.05, 0.95], [0.48, 0.48], color="black", lw=1.5)
ax.text(0.50, 0.52, "first arc: mechanism → operating regime", ha="center", fontsize=12)

bottom = [
    ("43\nResource\nAllocation", 0.34),
    ("49\nAdaptive\nInfrastructure", 0.50),
    ("53\nDistributed\nScheduling", 0.66),
    ("59\nCluster\nOptimization", 0.82)
]
for i, (label, x) in enumerate(bottom):
    rounded_box(ax, (x, 0.23), w, h, label, fontsize=10)
    if i < len(bottom)-1:
        arrow(ax, (x+w, 0.305), (bottom[i+1][1]-0.01, 0.305), ms=10)

ax.text(0.62, 0.10, "second arc: controller → infrastructure → distributed scheduling", ha="center", fontsize=12)
savefig("49_repository_transition.png")

## 2. Infrastructure map

The controller becomes useful only when its decisions reach concrete system components: queues, routers, replicas, telemetry, and autoscaling.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive serving infrastructure map", pad=22)

# main path
boxes = [
    ("Requests", 0.05, 0.58, 0.13, 0.12),
    ("Ingress\nqueue", 0.22, 0.58, 0.13, 0.12),
    ("Routing\ncontroller", 0.40, 0.58, 0.16, 0.12),
    ("Replica\npool", 0.63, 0.58, 0.13, 0.12),
    ("Target\nverify", 0.82, 0.58, 0.13, 0.12),
]
for i, (label, x, y, w, h) in enumerate(boxes):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
    if i < len(boxes)-1:
        arrow(ax, (x+w, y+h/2), (boxes[i+1][1]-0.01, y+h/2))

# control components
rounded_box(ax, (0.38, 0.30), 0.20, 0.11, "Operating-regime\nclassifier", fontsize=12)
rounded_box(ax, (0.15, 0.30), 0.15, 0.10, "Telemetry\nstate", fontsize=12)
rounded_box(ax, (0.65, 0.30), 0.15, 0.10, "Autoscaler", fontsize=12)
rounded_box(ax, (0.44, 0.12), 0.12, 0.09, "Policy\ncache", fontsize=11)

arrow(ax, (0.475, 0.58), (0.48, 0.41))
arrow(ax, (0.30, 0.35), (0.38, 0.35))
arrow(ax, (0.58, 0.35), (0.65, 0.35))
arrow(ax, (0.72, 0.40), (0.70, 0.58))
arrow(ax, (0.50, 0.30), (0.50, 0.21))
arrow(ax, (0.56, 0.165), (0.63, 0.58), connectionstyle="arc3,rad=-0.35")
ax.text(0.5, 0.04, "Infrastructure turns controller policy into routing, scaling, and verification placement.", ha="center", fontsize=13)
savefig("49_infrastructure_map.png")

## 3. Runtime routing controller

Routing is the place where regime information enters request handling. The same incoming request can receive different draft, verification, or replica treatment under different serving states.

In [ ]:
flow_diagram(
    "Runtime routing controller",
    ["request\nmetadata", "confidence\nprofile", "system\nstate", "policy\nlookup", "route\nassignment", "replica\nexecution"],
    "49_runtime_routing_controller.png",
    caption="Routing conditions on both request structure and live infrastructure state.",
    figsize=(15, 5.2)
)

## 4. Telemetry feedback loop

The serving system becomes adaptive when telemetry feeds back into the state estimator and controller.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Telemetry feedback closes the infrastructure loop", pad=20)

nodes = {
    "Request\nstream": (0.12, 0.70, 0.15, 0.10),
    "State\nestimator": (0.42, 0.70, 0.16, 0.10),
    "Routing /\nscheduling": (0.72, 0.70, 0.16, 0.10),
    "Serving\nsystem": (0.72, 0.34, 0.16, 0.10),
    "Telemetry\ncollector": (0.42, 0.34, 0.16, 0.10),
    "Policy\nupdate": (0.12, 0.34, 0.15, 0.10),
}
for label, (x,y,w,h) in nodes.items():
    rounded_box(ax, (x,y), w,h, label, fontsize=12)

arrow(ax, (0.27,0.75), (0.42,0.75))
arrow(ax, (0.58,0.75), (0.72,0.75))
arrow(ax, (0.80,0.70), (0.80,0.44))
arrow(ax, (0.72,0.39), (0.58,0.39))
arrow(ax, (0.42,0.39), (0.27,0.39))
arrow(ax, (0.20,0.44), (0.20,0.70))
arrow(ax, (0.50,0.44), (0.50,0.70), connectionstyle="arc3,rad=-0.18")
ax.text(0.5, 0.15, "Observed latency, memory pressure, queue depth, and acceptance rate update the next policy.", ha="center", fontsize=13)
savefig("49_telemetry_feedback_loop.png")

## 5. Autoscaling under regime changes

Scaling gains are regime dependent: balanced regimes benefit quickly, compute-limited regimes need more replicas, and latency-protected regimes saturate early.

In [ ]:
replicas = np.arange(1, 13)
balanced = 1.0 - np.exp(-0.42 * replicas)
compute = 0.92 * (1.0 - np.exp(-0.23 * replicas))
memory = 0.72 * (1.0 - np.exp(-0.55 * replicas))
latency = 0.60 * (1.0 - np.exp(-0.80 * replicas)) - 0.015 * np.maximum(replicas - 7, 0) ** 1.5

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(replicas, balanced, marker="o", label="balanced")
ax.plot(replicas, compute, marker="o", label="compute-limited")
ax.plot(replicas, memory, marker="o", label="memory-limited")
ax.plot(replicas, latency, marker="o", label="latency-protected")
ax.set_title("Autoscaling gains depend on operating regime")
ax.set_xlabel("allocated replicas")
ax.set_ylabel("throughput gain proxy")
ax.grid(alpha=0.3)
ax.legend()
savefig("49_autoscaling_regime_gains.png")

## 6. GPU pool allocation by policy

The resource allocator partitions GPU capacity across drafting, verification, reserve headroom, and telemetry overhead.

In [ ]:
regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
parts = ["draft", "verify", "reserve", "telemetry"]
data = np.array([
    [0.38, 0.34, 0.18, 0.10],
    [0.50, 0.25, 0.15, 0.10],
    [0.30, 0.36, 0.24, 0.10],
    [0.28, 0.22, 0.40, 0.10],
])

fig, ax = plt.subplots(figsize=(13, 6))
left = np.zeros(len(regimes))
for j, part in enumerate(parts):
    ax.barh(regimes, data[:, j], left=left, label=part)
    for i, val in enumerate(data[:, j]):
        if val > 0.08:
            ax.text(left[i] + val/2, i, f"{part}\n{val:.2f}", ha="center", va="center", fontsize=10)
    left += data[:, j]
ax.set_xlim(0, 1)
ax.set_xlabel("fraction of GPU pool")
ax.set_title("GPU pool allocation changes by controller policy")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=4)
ax.grid(axis="x", alpha=0.25)
savefig("49_gpu_pool_allocation.png")

## 7. Queue depth versus latency target

A routing controller must trade queue depth against verification length and target latency.

In [ ]:
queue = np.arange(4, 65, 4)
lat_targets = [0.8, 1.0, 1.3, 1.7]
fig, ax = plt.subplots(figsize=(12, 6))
for tau in lat_targets:
    sched = np.maximum(1, np.round(8 / (1 + (queue / (18 * tau))**1.4))).astype(int)
    ax.step(queue, sched, where="mid", marker="o", label=f"latency target={tau:.1f}")
ax.set_title("Queue depth shortens scheduled verification under tighter latency")
ax.set_xlabel("queue depth")
ax.set_ylabel("selected verification length ℓ*")
ax.set_ylim(0.5, 8.5)
ax.grid(alpha=0.3)
ax.legend()
savefig("49_queue_latency_schedule.png")

## 8. Replica routing by confidence and load

High-confidence traffic can be routed to faster paths; low-confidence traffic needs verification-heavy paths. Load shifts the boundary.

In [ ]:
confidence = np.linspace(0.2, 0.95, 80)
load = np.linspace(0.1, 1.0, 80)
C, L = np.meshgrid(confidence, load)

score = 0.60*C - 0.45*L + 0.18*np.sin(2.6*C)
policy = np.zeros_like(score, dtype=int)
policy[(score > -0.12) & (score <= 0.10)] = 1
policy[(score > 0.10) & (score <= 0.27)] = 2
policy[score > 0.27] = 3

labels = ["safe verify", "mixed route", "fast draft", "fast path"]
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(policy, origin="lower", extent=[confidence.min(), confidence.max(), load.min(), load.max()], aspect="auto")
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(labels)
cbar.set_label("selected route")
ax.set_title("Replica routing depends on confidence and active load")
ax.set_xlabel("request confidence")
ax.set_ylabel("active load")
ax.text(0.33, 0.78, "safe\nverify", ha="center", fontsize=12)
ax.text(0.55, 0.55, "mixed\nroute", ha="center", fontsize=12)
ax.text(0.77, 0.34, "fast\ndraft", ha="center", fontsize=12)
ax.text(0.88, 0.18, "fast\npath", ha="center", fontsize=12)
savefig("49_replica_routing_surface.png")

## 9. Infrastructure phase map

The active infrastructure phase is selected by load and available reserve capacity.

In [ ]:
reserve = np.linspace(0.05, 0.9, 100)
load = np.linspace(0.05, 1.0, 100)
R, L = np.meshgrid(reserve, load)

phase = np.zeros_like(R, dtype=int)
phase[(L > 0.45) & (R > 0.35)] = 1
phase[(L > 0.65) & (R <= 0.35)] = 2
phase[(L <= 0.45) & (R > 0.55)] = 3

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(phase, origin="lower", extent=[reserve.min(), reserve.max(), load.min(), load.max()], aspect="auto")
cbar = plt.colorbar(im, ticks=[0,1,2,3])
cbar.ax.set_yticklabels(["steady", "scale-out", "shed/shorten", "reserve"])
cbar.set_label("infrastructure phase")
ax.set_title("Infrastructure phase map")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
ax.text(0.25, 0.25, "steady", ha="center", fontsize=13)
ax.text(0.62, 0.72, "scale-out", ha="center", fontsize=13)
ax.text(0.18, 0.82, "shed /\nshorten", ha="center", fontsize=13)
ax.text(0.70, 0.25, "reserve", ha="center", fontsize=13)
savefig("49_infrastructure_phase_map.png")

## 10. Failure and saturation fallback

The infrastructure controller should expose a fallback path rather than silently degrading.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Saturation fallback path", pad=20)

main = [
    ("normal\nrouting", 0.06, 0.58),
    ("load\nspike", 0.24, 0.58),
    ("constraint\nactivation", 0.43, 0.58),
    ("shorten\nschedule", 0.64, 0.58),
    ("served\noutput", 0.82, 0.58),
]
w, h = 0.13, 0.12
for i, (label, x, y) in enumerate(main):
    rounded_box(ax, (x, y), w, h, label, fontsize=12)
    if i < len(main)-1:
        arrow(ax, (x+w, y+h/2), (main[i+1][1]-0.01, y+h/2))

rounded_box(ax, (0.43, 0.25), 0.17, 0.11, "fallback:\nqueue / defer", fontsize=12)
rounded_box(ax, (0.67, 0.25), 0.17, 0.11, "telemetry:\nmark degraded", fontsize=12)
arrow(ax, (0.495, 0.58), (0.515, 0.36))
arrow(ax, (0.60, 0.305), (0.67, 0.305))
arrow(ax, (0.755, 0.36), (0.71, 0.58), connectionstyle="arc3,rad=-0.2")
ax.text(0.5, 0.10, "Fallback makes saturation observable and recoverable.", ha="center", fontsize=13)
savefig("49_saturation_fallback_path.png")

## 11. Final synthesis

The infrastructure layer specifies how regime-conditioned policies become deployable routing and scaling behavior.

In [ ]:
flow_diagram(
    "Infrastructure specifies adaptive serving",
    ["operating\nregime", "controller\npolicy", "routing\nstate", "replica\nallocation", "telemetry\nfeedback", "adaptive\nserving"],
    "49_final_synthesis.png",
    caption="Notebook 49 bridges notebook-level scheduling policy and deployment-scale serving infrastructure.",
    figsize=(15, 5.2)
)

## Download generated figures

Run the cell below after executing the notebook to create a zip archive of all Notebook 49 figures.

In [ ]:
import zipfile
zip_path = Path("49_adaptive_infrastructure_figures.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FIG_DIR.glob("49_*.png")):
        zf.write(path, arcname=path.name)

display(FileLink(zip_path))

## Notebook summary

Notebook 49 completes the bridge from resource allocation to deployable infrastructure:

- controller policy becomes runtime routing;
- telemetry closes the loop;
- autoscaling and replica routing depend on operating regime;
- fallback paths make saturation observable;
- infrastructure turns confidence scheduling into an adaptive serving system.